In [1]:
import numpy as np
import pandas as pd

In [2]:
from transformers import AutoTokenizer
from transformers import RobertaForSequenceClassification, RobertaTokenizer
from transformers import pipeline

/Users/dylanhuang/micromamba/envs/df_ae2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch

In [4]:
from tqdm.notebook import tqdm

In [6]:
tweets = pd.read_parquet("../data/dataset/stock_tweets_sentiment_unifyemotion_nomerge.parquet")

In [7]:
tweets

,ticker,text,created_at,user_id,date,sentiment,positive_emotion,negative_emotion,uncertainty_emotion
0,AAPL,summary of yesterdays webcast featuring wynn g...,2013-12-31 23:10:08+00:00,1864753100,2013-12-31,4,0.021908,0.829146,0.144724
1,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 01:18:36+00:00,1937591882,2014-01-01,4,0.021908,0.829146,0.144724
2,AAPL,iphone users are more intelligent than samsung...,2014-01-01 01:52:31+00:00,23954327,2014-01-01,5,0.017082,0.944506,0.034343
3,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:29:29+00:00,1933063572,2014-01-01,4,0.021908,0.829146,0.144724
4,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:59:03+00:00,1938270918,2014-01-01,4,0.021908,0.829146,0.144724
...,...,...,...,...,...,...,...,...,...
106333,XOM,t active morning movers at t nyse t exxon mobil …,2015-12-28 17:15:13+00:00,2342763212,2015-12-28,5,0.018933,0.889814,0.079564
106334,XOM,divest from stopcommoncore optout because chil...,2015-12-28 19:39:46+00:00,4399710563,2015-12-28,1,0.021422,0.888479,0.077954
106335,XOM,zsl stock forum zsl gold uslv zsl investing na...,2015-12-29 16:52:36+00:00,2181314366,2015-12-29,5,0.045692,0.738872,0.202946
106336,XOM,nptn recent news updated tuesday december pm g...,2015-12-29 19:03:17+00:00,2197054086,2015-12-29,1,0.067945,0.815209,0.113054


In [8]:
tweets.shape

(106338, 9)

In [9]:
tokenizer_loaded = RobertaTokenizer.from_pretrained('zhayunduo/roberta-base-stocktwits-finetuned')
model_loaded = RobertaForSequenceClassification.from_pretrained('zhayunduo/roberta-base-stocktwits-finetuned')


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1859.22it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: zhayunduo/roberta-base-stocktwits-finetuned
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
bert_model = model_loaded.to(device)

In [11]:
torch.cuda.is_available()
print(next(bert_model.parameters()).device)

cpu


In [12]:
nlp = pipeline("text-classification", model=bert_model, tokenizer=tokenizer_loaded)

In [13]:
def stance_score(text):
    result = nlp(text)
    label = result[0]['label']
    score = result[0]['score']


    
    return label, score

In [14]:
tweets[['stance_label','stance_score']] = tweets['text'].apply(stance_score).apply(pd.Series)

In [15]:
tweets

,ticker,text,created_at,user_id,date,sentiment,positive_emotion,negative_emotion,uncertainty_emotion,stance_label,stance_score
0,AAPL,summary of yesterdays webcast featuring wynn g...,2013-12-31 23:10:08+00:00,1864753100,2013-12-31,4,0.021908,0.829146,0.144724,Positive,0.742110
1,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 01:18:36+00:00,1937591882,2014-01-01,4,0.021908,0.829146,0.144724,Positive,0.742110
2,AAPL,iphone users are more intelligent than samsung...,2014-01-01 01:52:31+00:00,23954327,2014-01-01,5,0.017082,0.944506,0.034343,Positive,0.998410
3,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:29:29+00:00,1933063572,2014-01-01,4,0.021908,0.829146,0.144724,Positive,0.742110
4,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:59:03+00:00,1938270918,2014-01-01,4,0.021908,0.829146,0.144724,Positive,0.742110
...,...,...,...,...,...,...,...,...,...,...,...
106333,XOM,t active morning movers at t nyse t exxon mobil …,2015-12-28 17:15:13+00:00,2342763212,2015-12-28,5,0.018933,0.889814,0.079564,Positive,0.803462
106334,XOM,divest from stopcommoncore optout because chil...,2015-12-28 19:39:46+00:00,4399710563,2015-12-28,1,0.021422,0.888479,0.077954,Negative,0.949051
106335,XOM,zsl stock forum zsl gold uslv zsl investing na...,2015-12-29 16:52:36+00:00,2181314366,2015-12-29,5,0.045692,0.738872,0.202946,Positive,0.997038
106336,XOM,nptn recent news updated tuesday december pm g...,2015-12-29 19:03:17+00:00,2197054086,2015-12-29,1,0.067945,0.815209,0.113054,Positive,0.997328


In [16]:
stance_score("buy buy buy!")

('Positive', 0.9862682819366455)

In [17]:
tweets.to_parquet('../data/dataset/stock_tweets_sentiment_unifyemotion_stanceScore_nomerge.parquet',index=False)